In [1]:
import warnings
import tensorflow as tf
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

import numpy as np
import pandas as pd
import deepchem as dc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_max_pool
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from sklearn import metrics

Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


In [2]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = GCNConv(30, 256)
        self.conv2 = GCNConv(256, 256)
        self.conv3 = GCNConv(256, 256)
        self.conv4 = GCNConv(256, 256)
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)
        self.dropout1 = nn.Dropout(p=0.4)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        x = self.conv4(x, edge_index)
        x = F.relu(x)
        x = global_max_pool(x, data.batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

In [3]:
def custom_collate(batch):
    data_list, target_list = zip(*batch)
    batch_data = Batch.from_data_list(data_list)
    batch_target = torch.stack(target_list)
    return batch_data, batch_target

In [4]:
def calculate_statistics(group):
    r2_test = group['r2_test']
    r2_test_dict = {f'run{i}': r2_test_val for i, r2_test_val in enumerate(r2_test)}
    return pd.Series({
        **r2_test_dict, 
        'r2_test_mean': np.mean(r2_test),
        'r2_test_max': np.max(r2_test),
        'r2_test_min': np.min(r2_test),
        'r2_test_std': np.std(r2_test, ddof=0),
    })

def calculate_statistics2(group):
    rmse_test = group['rmse_test']
    rmse_test_dict = {f'run{i}': rmse_test_val for i, rmse_test_val in enumerate(rmse_test)}
    return pd.Series({
        **rmse_test_dict, 
        'rmse_test_mean': np.mean(rmse_test),
        'rmse_test_max': np.max(rmse_test),
        'rmse_test_min': np.min(rmse_test),
        'rmse_test_std': np.std(rmse_test, ddof=0),
    })

In [5]:
torch.manual_seed(0)

epochs = 120
lr = 9e-3
wd = 1e-5

results_r2 = []
results_rmse = []
for random_state in range(10):
    torch.manual_seed(0)
    
    for dataset in ["abcgg", "aatsc3d", "atsc3d", "kappa2", "peoevsa6", "bertzct", "ggi10", "vsaestate3",
                    "atsc4i", "bcutp1l", "kappa3", "estatevsa3", "kier3", "aats8p", "kier2", "frnh0"]:
        torch.manual_seed(0)
        
        for t in ["Yield_CA"]:
            torch.manual_seed(0)
            scaler = StandardScaler()
            df = pd.read_csv('data_Real/data_real.csv')
            smiles = df["SMILES"]
            featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
            X = featurizer.featurize(smiles)
            
            y = df[t]
            data_train, data_test, target_train, target_test = train_test_split(X, y, test_size=0.5, random_state=random_state)

            target_train = scaler.fit_transform(target_train.values.reshape(-1, 1)).flatten()
            target_test = scaler.transform(target_test.values.reshape(-1, 1)).flatten()
            
            target_train = torch.tensor(target_train, dtype=torch.float32)
            target_test = torch.tensor(target_test, dtype=torch.float32)

            data_train_list = []
            for graph_data in data_train:
                node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
                edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
                edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
                data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
                data_train_list.append(data)

            data_test_list = []
            for graph_data in data_test:
                node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
                edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
                edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
                data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
                data_test_list.append(data)

            train_loader = DataLoader(list(zip(data_train_list, target_train)), batch_size=len(data_train_list), collate_fn=custom_collate)
            test_loader = DataLoader(list(zip(data_test_list, target_test)), batch_size=len(data_test_list), collate_fn=custom_collate)

            model = Net()
            model.load_state_dict(torch.load(f'data_AI2+Human/model_{dataset}_sc.pth'))
            model.fc3 = nn.Linear(128, 1)
        
            model.train()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
            criterion = nn.MSELoss()
        
            for param in model.conv1.parameters():
                param.requires_grad = False
            for param in model.conv2.parameters():
                param.requires_grad = False
            for param in model.conv3.parameters():
                param.requires_grad = False
            for param in model.conv4.parameters():
                param.requires_grad = False

            device = torch.device('cpu')
            model.to(device)

            for epoch in range(epochs):
                for data, target in train_loader:
                    data = data.to(device)
                    target = target.to(device)
                    optimizer.zero_grad()
                    out = model(data)
                    loss = criterion(out, target.view(-1, 1))
                    loss.backward()
                    optimizer.step()

            model.eval()
            pred_train = []
            for data, target in train_loader:
                data = data.to(device)
                with torch.no_grad():
                    out = model(data)
                pred_train.append(out.cpu().numpy())
            pred_train = np.concatenate(pred_train)

            pred_test = []
            for data, target in test_loader:
                data = data.to(device)
                with torch.no_grad():
                    out = model(data)
                pred_test.append(out.cpu().numpy())
            pred_test = np.concatenate(pred_test)

            pred_train = scaler.inverse_transform(pred_train)
            pred_test = scaler.inverse_transform(pred_test)
            target_train = scaler.inverse_transform(target_train.numpy().reshape(-1, 1)).flatten()
            target_test = scaler.inverse_transform(target_test.numpy().reshape(-1, 1)).flatten()

            r2_test_score = metrics.r2_score(target_test, pred_test)
            rmse_test_score = metrics.root_mean_squared_error(target_test, pred_test)
            results_r2.append({'source': dataset, 'target': t, 'r2_test': r2_test_score})
            results_rmse.append({'source': dataset, 'target': t, 'rmse_test': rmse_test_score})

results_df = pd.DataFrame(results_r2)
gen_results = results_df.groupby(['source', 'target']).apply(calculate_statistics).reset_index()
results_df2 = pd.DataFrame(results_rmse)
gen_results2 = results_df2.groupby(['source', 'target']).apply(calculate_statistics2).reset_index()

In [6]:
gen_results.T.to_csv('result/result_yield_CA_r2.csv', header=False)
gen_results.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
source,aats8p,aatsc3d,abcgg,atsc3d,atsc4i,bcutp1l,bertzct,estatevsa3,frnh0,ggi10,kappa2,kappa3,kier2,kier3,peoevsa6,vsaestate3
target,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA
run0,-0.048752,-0.055231,0.288653,-0.095357,-0.037124,-0.515108,0.158251,-0.484548,-0.104237,0.131882,-0.085108,0.190561,0.131592,0.047409,-0.10339,-0.28486
run1,0.317502,0.432392,0.725024,0.505782,0.519744,0.085257,0.470441,0.516986,0.365394,0.364658,0.579373,0.526794,0.449342,0.474055,0.413531,0.302329
run2,0.287295,0.307919,0.363912,0.305498,0.289579,0.387801,0.365655,0.319387,0.32903,0.39399,0.346454,0.459508,0.313917,0.339306,0.338629,0.198272
run3,0.206394,0.207198,0.304595,0.368238,0.166854,0.127883,0.260391,-0.021111,0.333944,0.230135,0.22888,0.214012,0.229667,0.269337,0.141018,-0.031567
run4,0.295514,0.240325,0.449816,0.263187,0.191491,0.071498,0.263477,0.059342,0.275921,0.159476,0.267377,0.349696,0.27084,0.27739,0.301124,0.056247
run5,0.497487,0.501753,0.527761,0.491959,0.336753,-0.012843,0.442584,0.312231,0.476146,0.477804,0.412337,0.536925,0.381338,0.421274,0.439012,0.394523
run6,0.200067,0.103685,0.264607,0.174701,0.102145,0.038769,0.272594,0.197344,0.172811,0.232486,0.249473,0.244557,0.149984,0.323467,0.268986,0.323832
run7,0.424346,0.512004,0.633866,0.500975,0.371414,0.221942,0.519006,0.036237,0.317352,0.292019,0.466948,0.56098,0.480992,0.528338,0.513051,0.407295


In [7]:
gen_results2.T.to_csv('result/result_yield_CA_rmse.csv', header=False)
gen_results2.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
source,aats8p,aatsc3d,abcgg,atsc3d,atsc4i,bcutp1l,bertzct,estatevsa3,frnh0,ggi10,kappa2,kappa3,kier2,kier3,peoevsa6,vsaestate3
target,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA,Yield_CA
run0,22.759268,22.829468,18.744024,23.259472,22.632751,27.355446,20.389818,27.078159,23.353563,20.706722,23.150396,19.994663,20.710176,21.690783,23.344603,25.191259
run1,18.375013,16.757185,11.663383,15.636379,15.413925,21.27289,16.185799,15.45812,17.71858,17.728861,14.425326,15.300371,16.505083,16.130465,17.033321,18.578135
run2,22.045517,21.724199,20.826862,21.762165,22.010153,20.432035,20.798319,21.543461,21.390299,20.328501,21.110737,19.198172,21.629862,21.22587,21.236738,23.381845
run3,22.167395,22.156157,20.750624,19.778292,22.7129,23.238043,21.399973,25.144821,20.308022,21.8333,21.851086,22.060738,21.839931,21.270153,23.062386,25.273239
run4,21.938587,22.781719,19.387707,22.436293,23.502544,25.186283,22.431883,25.350607,22.241571,23.963358,22.372417,21.07807,22.319479,22.218996,21.851057,25.392277
run5,19.124357,19.043018,18.539335,19.229259,21.971064,27.150911,20.142029,22.373535,19.526228,19.495298,20.681292,18.358582,21.21973,20.523415,20.206459,20.992405
run6,21.893703,23.175152,20.991913,22.238113,23.195061,23.99972,20.877604,21.930931,22.263567,21.445467,21.206816,21.276152,22.568668,20.134314,20.929319,20.128883
run7,20.495663,18.870762,16.345617,19.082802,21.417242,23.82798,18.73488,26.519562,22.31925,22.729612,19.722677,17.898783,19.461138,18.552248,18.850504,20.796989
